In [54]:
import json                                                                                                  
import pandas as pd
from kafka import KafkaProducer
from time import time

In [55]:
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet"                         
columns = ['lpep_pickup_datetime', 'lpep_dropoff_datetime', 'PULocationID', 'DOLocationID',                  
           'passenger_count', 'trip_distance', 'tip_amount', 'total_amount']                                 
df = pd.read_parquet(url, columns=columns)

In [56]:
producer = KafkaProducer(                                                                                    
    bootstrap_servers='localhost:9092',                                                                      
    value_serializer=lambda v: json.dumps(v).encode('utf-8')                                                 
)

In [ ]:
t0 = time()

for _, row in df.iterrows():
    row['lpep_pickup_datetime'] = str(row['lpep_pickup_datetime'])
    row['lpep_dropoff_datetime'] = str(row['lpep_dropoff_datetime'])                                         
    producer.send('green-trips', value=row.to_dict())

producer.flush()                                                                                             
t1 = time()
print(f'took {(t1 - t0):.2f} seconds')